In [1]:
# Section 1: Import Required Libraries
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


In [2]:
# Section 2: Load the Spambase Dataset
def load_spambase_data(filepath='spambase.data'):
    data = pd.read_csv(filepath, header=None)
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    return X, y

# Load the dataset
X, y = load_spambase_data('spambase.data')

print("✓ Dataset loaded successfully!")
print(f"  - Number of samples: {X.shape[0]}")
print(f"  - Number of features: {X.shape[1]}")
print(f"  - Non-spam emails: {np.sum(y == 0)}")
print(f"  - Spam emails: {np.sum(y == 1)}")

✓ Dataset loaded successfully!
  - Number of samples: 4601
  - Number of features: 57
  - Non-spam emails: 2788
  - Spam emails: 1813


In [3]:
# Section 3: Statistical Test Functions (Manual Implementation)

def friedman_test(data):
    """Friedman test - manually implemented"""
    n_datasets, k = data.shape
    ranks = np.zeros_like(data)
    
    for i in range(n_datasets):
        ranks[i] = stats.rankdata(-data[i])
    
    avg_ranks = np.mean(ranks, axis=0)
    rank_sum_sq = np.sum(avg_ranks ** 2)
    chi2_f = (12 * n_datasets / (k * (k + 1))) * (rank_sum_sq - k * ((k + 1) ** 2) / 4)
    df = k - 1
    p_value = 1 - stats.chi2.cdf(chi2_f, df)
    
    return chi2_f, p_value, avg_ranks

def nemenyi_critical_difference(n_datasets, k, alpha=0.05):
    """Nemenyi critical difference - manually implemented"""
    q_alpha_values = {
        2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
        7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164
    }
    q_alpha = q_alpha_values[k]
    cd = q_alpha * np.sqrt((k * (k + 1)) / (6 * n_datasets))
    return cd

print("✓ Statistical test functions defined!")

✓ Statistical test functions defined!


In [4]:
# Section 4: Run 10-Fold Cross-Validation Experiments

def evaluate_algorithms(X, y, n_folds=10):
    # Initialize algorithms
    algorithms = {
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Naive Bayes': GaussianNB(),
        'SVM': SVC(random_state=42)
    }
    
    # Initialize result storage
    results = {
        'training_time': {name: [] for name in algorithms.keys()},
        'accuracy': {name: [] for name in algorithms.keys()},
        'f_measure': {name: [] for name in algorithms.keys()}
    }
    
    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_num = 1
    for train_idx, test_idx in skf.split(X, y):
        print(f"Processing Fold {fold_num}/{n_folds}...", end=' ')
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Evaluate each algorithm
        for name, clf in algorithms.items():
            start_time = time.time()
            clf.fit(X_train_scaled, y_train)
            training_time = time.time() - start_time
            
            y_pred = clf.predict(X_test_scaled)
            
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='binary')
            
            results['training_time'][name].append(training_time)
            results['accuracy'][name].append(acc)
            results['f_measure'][name].append(f1)
        
        print("✓")
        fold_num += 1
    
    return results

print("="*60)
print("RUNNING 10-FOLD CROSS-VALIDATION")
print("="*60)
results = evaluate_algorithms(X, y)
print("\n✓ All experiments completed!")

RUNNING 10-FOLD CROSS-VALIDATION
Processing Fold 1/10... ✓
Processing Fold 2/10... ✓
Processing Fold 3/10... ✓
Processing Fold 4/10... ✓
Processing Fold 5/10... ✓
Processing Fold 6/10... ✓
Processing Fold 7/10... ✓
Processing Fold 8/10... ✓
Processing Fold 9/10... ✓
Processing Fold 10/10... ✓

✓ All experiments completed!


In [5]:
# Section 5: Training Time Results

print("="*70)
print("METRIC 1: TRAINING TIME (seconds)")
print("="*70)

# Create results table
results_df_time = pd.DataFrame(results['training_time'])
results_df_time.index = [f'Fold {i+1}' for i in range(10)]

print("\n📊 Results Table:")
print(results_df_time.to_string())

# Statistical analysis
data_array = results_df_time.values
rank_data = -data_array  # Lower time is better

algorithm_names = list(results['training_time'].keys())
chi2_f, p_value, avg_ranks = friedman_test(rank_data)

print(f"\n\nFriedman Test Results:")
print(f"  χ²_F = {chi2_f:.4f}")
print(f"  p-value = {p_value:.6f}")
print(f"\nAverage Ranks:")
for name, rank in zip(algorithm_names, avg_ranks):
    print(f"  {name}: {rank:.2f}")

# Check significance
alpha = 0.05
is_significant = p_value < alpha
print(f"\nSignificance (α = {alpha}):")
if is_significant:
    print(f"  SIGNIFICANT - p-value ({p_value:.6f}) < {alpha}")
    cd = nemenyi_critical_difference(10, 3, alpha)
    print(f"\nNemenyi Critical Difference: {cd:.4f}")
    print("\nPairwise Comparisons:")
    for i in range(3):
        for j in range(i+1, 3):
            diff = abs(avg_ranks[i] - avg_ranks[j])
            sig = "SIGNIFICANTLY DIFFERENT" if diff > cd else "Not significantly different"
            print(f"  {algorithm_names[i]} vs {algorithm_names[j]}: {diff:.4f} → {sig}")
else:
    print(f"  NOT SIGNIFICANT - p-value ({p_value:.6f}) >= {alpha}")

# Save to CSV
results_df_time.to_csv('results_training_time.csv')
print(f"\n💾 Saved: results_training_time.csv")

METRIC 1: TRAINING TIME (seconds)

📊 Results Table:
         Decision Tree  Naive Bayes       SVM
Fold 1        0.099025     0.000000  0.236691
Fold 2        0.074432     0.000000  0.214832
Fold 3        0.052112     0.012015  0.231978
Fold 4        0.046818     0.016636  0.223377
Fold 5        0.060555     0.000000  0.241619
Fold 6        0.056068     0.000000  0.234289
Fold 7        0.046372     0.009641  0.224457
Fold 8        0.069211     0.010385  0.221146
Fold 9        0.062218     0.004015  0.217307
Fold 10       0.053552     0.000000  0.246213


Friedman Test Results:
  χ²_F = 20.0000
  p-value = 0.000045

Average Ranks:
  Decision Tree: 2.00
  Naive Bayes: 1.00
  SVM: 3.00

Significance (α = 0.05):
  SIGNIFICANT - p-value (0.000045) < 0.05

Nemenyi Critical Difference: 1.0478

Pairwise Comparisons:
  Decision Tree vs Naive Bayes: 1.0000 → Not significantly different
  Decision Tree vs SVM: 1.0000 → Not significantly different
  Naive Bayes vs SVM: 2.0000 → SIGNIFICANTLY DIFFER

In [6]:
# Section 6: Accuracy Results

print("="*70)
print("METRIC 2: ACCURACY")
print("="*70)

# Create results table
results_df_acc = pd.DataFrame(results['accuracy'])
results_df_acc.index = [f'Fold {i+1}' for i in range(10)]

print("\n📊 Results Table:")
print(results_df_acc.to_string())

# Statistical analysis
data_array = results_df_acc.values
rank_data = data_array  # Higher accuracy is better

algorithm_names = list(results['accuracy'].keys())
chi2_f, p_value, avg_ranks = friedman_test(rank_data)

print(f"\n\nFriedman Test Results:")
print(f"  χ²_F = {chi2_f:.4f}")
print(f"  p-value = {p_value:.6f}")
print(f"\nAverage Ranks:")
for name, rank in zip(algorithm_names, avg_ranks):
    print(f"  {name}: {rank:.2f}")

# Check significance
alpha = 0.05
is_significant = p_value < alpha
print(f"\nSignificance (α = {alpha}):")
if is_significant:
    print(f"  SIGNIFICANT - p-value ({p_value:.6f}) < {alpha}")
    cd = nemenyi_critical_difference(10, 3, alpha)
    print(f"\nNemenyi Critical Difference: {cd:.4f}")
    print("\nPairwise Comparisons:")
    for i in range(3):
        for j in range(i+1, 3):
            diff = abs(avg_ranks[i] - avg_ranks[j])
            sig = "SIGNIFICANTLY DIFFERENT" if diff > cd else "Not significantly different"
            print(f"  {algorithm_names[i]} vs {algorithm_names[j]}: {diff:.4f} → {sig}")
else:
    print(f"  NOT SIGNIFICANT - p-value ({p_value:.6f}) >= {alpha}")

# Save to CSV
results_df_acc.to_csv('results_accuracy.csv')
print(f"\n💾 Saved: results_accuracy.csv")

METRIC 2: ACCURACY

📊 Results Table:
         Decision Tree  Naive Bayes       SVM
Fold 1        0.878525     0.822126  0.919740
Fold 2        0.906522     0.819565  0.930435
Fold 3        0.921739     0.795652  0.939130
Fold 4        0.945652     0.813043  0.926087
Fold 5        0.904348     0.815217  0.936957
Fold 6        0.904348     0.789130  0.947826
Fold 7        0.913043     0.832609  0.926087
Fold 8        0.919565     0.823913  0.936957
Fold 9        0.921739     0.841304  0.945652
Fold 10       0.895652     0.813043  0.934783


Friedman Test Results:
  χ²_F = 18.2000
  p-value = 0.000112

Average Ranks:
  Decision Tree: 1.90
  Naive Bayes: 3.00
  SVM: 1.10

Significance (α = 0.05):
  SIGNIFICANT - p-value (0.000112) < 0.05

Nemenyi Critical Difference: 1.0478

Pairwise Comparisons:
  Decision Tree vs Naive Bayes: 1.1000 → SIGNIFICANTLY DIFFERENT
  Decision Tree vs SVM: 0.8000 → Not significantly different
  Naive Bayes vs SVM: 1.9000 → SIGNIFICANTLY DIFFERENT

💾 Saved: resul

In [7]:
# Section 7: F-measure Results

print("="*70)
print("METRIC 3: F-MEASURE")
print("="*70)

# Create results table
results_df_f1 = pd.DataFrame(results['f_measure'])
results_df_f1.index = [f'Fold {i+1}' for i in range(10)]

print("\n📊 Results Table:")
print(results_df_f1.to_string())

# Statistical analysis
data_array = results_df_f1.values
rank_data = data_array  # Higher F-measure is better

algorithm_names = list(results['f_measure'].keys())
chi2_f, p_value, avg_ranks = friedman_test(rank_data)

print(f"\n\nFriedman Test Results:")
print(f"  χ²_F = {chi2_f:.4f}")
print(f"  p-value = {p_value:.6f}")
print(f"\nAverage Ranks:")
for name, rank in zip(algorithm_names, avg_ranks):
    print(f"  {name}: {rank:.2f}")

# Check significance
alpha = 0.05
is_significant = p_value < alpha
print(f"\nSignificance (α = {alpha}):")
if is_significant:
    print(f"  SIGNIFICANT - p-value ({p_value:.6f}) < {alpha}")
    cd = nemenyi_critical_difference(10, 3, alpha)
    print(f"\nNemenyi Critical Difference: {cd:.4f}")
    print("\nPairwise Comparisons:")
    for i in range(3):
        for j in range(i+1, 3):
            diff = abs(avg_ranks[i] - avg_ranks[j])
            sig = "SIGNIFICANTLY DIFFERENT" if diff > cd else "Not significantly different"
            print(f"  {algorithm_names[i]} vs {algorithm_names[j]}: {diff:.4f} → {sig}")
else:
    print(f"  NOT SIGNIFICANT - p-value ({p_value:.6f}) >= {alpha}")

# Save to CSV
results_df_f1.to_csv('results_f_measure.csv')
print(f"\n💾 Saved: results_f_measure.csv")

METRIC 3: F-MEASURE

📊 Results Table:
         Decision Tree  Naive Bayes       SVM
Fold 1        0.852632     0.807512  0.896359
Fold 2        0.878873     0.806527  0.910112
Fold 3        0.901099     0.787330  0.922222
Fold 4        0.931507     0.801843  0.904494
Fold 5        0.882979     0.804598  0.919220
Fold 6        0.879781     0.781038  0.932584
Fold 7        0.889503     0.816229  0.903955
Fold 8        0.898072     0.806683  0.918310
Fold 9        0.901639     0.827423  0.928367
Fold 10       0.868132     0.801843  0.914773


Friedman Test Results:
  χ²_F = 18.2000
  p-value = 0.000112

Average Ranks:
  Decision Tree: 1.90
  Naive Bayes: 3.00
  SVM: 1.10

Significance (α = 0.05):
  SIGNIFICANT - p-value (0.000112) < 0.05

Nemenyi Critical Difference: 1.0478

Pairwise Comparisons:
  Decision Tree vs Naive Bayes: 1.1000 → SIGNIFICANTLY DIFFERENT
  Decision Tree vs SVM: 0.8000 → Not significantly different
  Naive Bayes vs SVM: 1.9000 → SIGNIFICANTLY DIFFERENT

💾 Saved: resu